# Classificazione Spettrale delle Stelle

La **classificazione spettrale stellare** (sistema Morgan-Keenan, MK) suddivide le stelle in classi in base alla loro temperatura superficiale. Le classi principali sono:

- **O** (>= 33 000 K): blu, molto massicce e luminose
- **B** (10 000 - 33 000 K): blu-bianche
- **A** (7 500 - 10 000 K): bianche (es. Vega)
- **F** (6 000 - 7 500 K): bianco-gialle (es. Polare)
- **G** (5 200 - 6 000 K): gialle (es. il Sole)
- **K** (3 700 - 5 200 K): arancioni
- **M** (2 000 - 3 700 K): rosse, le più comuni
- **L** (1 200 - 2 000 K): nane brune
- **T** (700 - 1 300 K): nane brune con metano

In questo notebook carichiamo un dataset stellare della NASA e classifichiamo le stelle in base alla temperatura.

## 1. Import delle librerie

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Caricamento del dataset

Il file `STAR_DATASET_2024.csv` contiene righe di commento (prefissate con `#`) seguite da dati separati da virgola. Usiamo `comment='#'` per ignorare automaticamente i commenti.

In [ ]:
def carica_dataset_stelle(percorso_file):
    """Carica il dataset stellare saltando le righe di commento.

    Parametri:
        percorso_file: percorso del file CSV

    Restituisce:
        DataFrame pandas con i dati, oppure None in caso di errore.
    """
    print("Directory corrente:", os.getcwd())
    print("Il file esiste?", os.path.exists(percorso_file))

    try:
        df = pd.read_csv(
            percorso_file,
            comment='#',
            on_bad_lines='skip',
            encoding='utf-8',
            low_memory=False
        )
        print(f"Dataset caricato: {len(df)} righe x {len(df.columns)} colonne")
        return df
    except Exception as e:
        print(f"Errore durante il caricamento: {e}")
        return None


file_path = './data/STAR_DATASET_2024.csv'
df_stelle = carica_dataset_stelle(file_path)

## 3. Identificazione della colonna temperatura

Cerchiamo la colonna che contiene la temperatura superficiale. I nomi possibili includono `hoststar_temperature`, `st_teff`, `teff`, ecc.

In [ ]:
def trova_colonna_temperatura(df):
    """Cerca la colonna della temperatura tra i nomi più comuni."""
    candidati = ['hoststar_temperature', 'st_teff', 'teff', 'temperature', 'temp']
    for col in df.columns:
        if col.lower() in candidati:
            return col
    # Se non trovata, stampa le prime colonne per ispezione manuale
    print("Colonna temperatura non trovata. Colonne disponibili:", df.columns[:15].tolist())
    return None


colonna_temp = trova_colonna_temperatura(df_stelle)
if colonna_temp:
    print(f"Colonna temperatura identificata: '{colonna_temp}'")
    print(f"Valori non nulli: {df_stelle[colonna_temp].notna().sum()}")

## 4. Funzione di classificazione spettrale

Definiamo una funzione che assegna la classe spettrale in base alla temperatura in Kelvin.

In [ ]:
def classifica_stella(temperatura):
    """Classifica una stella in base alla temperatura superficiale (K).

    Restituisce la lettera della classe spettrale secondo il sistema MK.
    """
    if pd.isna(temperatura):
        return 'Sconosciuta'
    if temperatura >= 33000:
        return 'O'
    if temperatura >= 10000:
        return 'B'
    if temperatura >= 7500:
        return 'A'
    if temperatura >= 6000:
        return 'F'
    if temperatura >= 5200:
        return 'G'
    if temperatura >= 3700:
        return 'K'
    if temperatura >= 2000:
        return 'M'
    if temperatura >= 1200:
        return 'L'
    if temperatura >= 700:
        return 'T'
    return 'Sconosciuta'


# Applichiamo la classificazione
if colonna_temp and colonna_temp in df_stelle.columns:
    df_stelle['classe_spettrale'] = df_stelle[colonna_temp].apply(classifica_stella)
    print("Classificazione completata.")
    print(df_stelle[[colonna_temp, 'classe_spettrale']].head(10))

## 5. Distribuzione delle classi spettrali

Visualizziamo quante stelle di ogni classe sono presenti nel dataset. Le stelle di classe M (fredde e poco luminose) sono tipicamente le più numerose.

In [ ]:
def grafico_distribuzione_spettrale(df):
    """Crea un countplot della distribuzione delle classi spettrali."""
    if 'classe_spettrale' not in df.columns:
        print("Colonna 'classe_spettrale' non trovata.")
        return None

    # Ordine corretto delle classi
    ordine_classi = ['O', 'B', 'A', 'F', 'G', 'K', 'M', 'L', 'T', 'Sconosciuta']
    classi_presenti = [c for c in ordine_classi if c in df['classe_spettrale'].values]

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.countplot(data=df, x='classe_spettrale', order=classi_presenti, palette='viridis', ax=ax)

    ax.set_title('Distribuzione delle Classi Spettrali delle Stelle', fontsize=14, fontweight='bold')
    ax.set_xlabel('Classe Spettrale', fontsize=12)
    ax.set_ylabel('Numero di Stelle', fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
    return fig


if df_stelle is not None:
    grafico_distribuzione_spettrale(df_stelle)